In [1]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import time
import pandas as pd
from skimage.metrics import mean_squared_error, peak_signal_noise_ratio, structural_similarity
from skimage.color import deltaE_ciede2000, rgb2lab
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set style for better plots
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Create directories if they don't exist
os.makedirs("Result_images", exist_ok=True)
os.makedirs("evaluation_results", exist_ok=True)
os.makedirs("Colorization_Dataset/baseline_outputs", exist_ok=True)
os.makedirs("Colorization_Dataset/evaluation_results", exist_ok=True)

In [2]:
# Load the colorization model
prototxt_path = "models/models_colorization_deploy_v2.prototxt"
model_path = "models/colorization_release_v2.caffemodel"
points_path = "models/pts_in_hull.npy"

def load_colorization_model():
    """Load and configure the colorization model"""
    net = cv2.dnn.readNetFromCaffe(prototxt_path, model_path)
    pts = np.load(points_path)
    
    class8 = net.getLayerId("class8_ab")
    conv8 = net.getLayerId("conv8_313_rh")
    
    pts = pts.transpose().reshape(2, 313, 1, 1)
    net.getLayer(class8).blobs = [pts.astype("float32")]
    net.getLayer(conv8).blobs = [np.full([1, 313], 2.606, dtype="float32")]
    
    return net

net = load_colorization_model()
print("Model loaded successfully!")

Model loaded successfully!


In [3]:
def colorize_image_with_metrics(image_path, net, compute_metrics=True, original_path=None):
    """
    Colorize an image and compute evaluation metrics if original is provided.
    """
    # Read image
    image_bgr = cv2.imread(image_path)
    if image_bgr is None:
        raise FileNotFoundError(f"Could not read image: {image_path}")
    
    start_time = time.time()
    
    # Convert original to RGB
    original_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    
    # Convert to grayscale
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    gray_bgr = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
    gray_rgb = cv2.cvtColor(gray_bgr, cv2.COLOR_BGR2RGB)
    
    # Colorization process
    scaled = gray_rgb.astype("float32") / 255.0
    lab = cv2.cvtColor(scaled, cv2.COLOR_RGB2LAB)
    resized = cv2.resize(lab, (224, 224))
    L = cv2.split(resized)[0]
    L -= 50
    
    net.setInput(cv2.dnn.blobFromImage(L))
    ab = net.forward()[0, :, :, :].transpose((1, 2, 0))
    ab = cv2.resize(ab, (gray_rgb.shape[1], gray_rgb.shape[0]))
    
    original_L = cv2.split(lab)[0]
    colorized_lab = np.concatenate((original_L[:, :, np.newaxis], ab), axis=2)
    colorized_rgb = cv2.cvtColor(colorized_lab, cv2.COLOR_LAB2RGB)
    colorized_rgb = np.clip(colorized_rgb, 0, 1)
    colorized_rgb = (colorized_rgb * 255).astype("uint8")
    
    inference_time = (time.time() - start_time) * 1000  # in milliseconds
    
    # Compute metrics if original image is provided
    metrics = {}
    if compute_metrics and original_path is not None:
        # Load original image
        original_compare = cv2.imread(original_path)
        original_compare = cv2.cvtColor(original_compare, cv2.COLOR_BGR2RGB)
        
        # Resize to match if needed
        if original_compare.shape[:2] != colorized_rgb.shape[:2]:
            original_compare = cv2.resize(original_compare, 
                                         (colorized_rgb.shape[1], colorized_rgb.shape[0]))
        
        # Convert to float for metrics
        original_float = original_compare.astype("float32") / 255.0
        colorized_float = colorized_rgb.astype("float32") / 255.0
        
        # MAE
        mae = np.mean(np.abs(original_float - colorized_float))
        metrics['MAE'] = mae
        
        # MSE
        mse = mean_squared_error(original_float, colorized_float)
        metrics['MSE'] = mse
        
        # PSNR
        psnr = peak_signal_noise_ratio(original_float, colorized_float, data_range=1)
        metrics['PSNR'] = psnr
        
        # SSIM
        ssim = structural_similarity(original_float, colorized_float, 
                                     channel_axis=2, data_range=1)
        metrics['SSIM'] = ssim
        
        # Delta E (perceptual color difference)
        lab_original = rgb2lab(original_float)
        lab_colorized = rgb2lab(colorized_float)
        delta_e = np.mean(deltaE_ciede2000(lab_original, lab_colorized))
        metrics['Delta E'] = delta_e
        
        metrics['Inference_Time_ms'] = inference_time
    
    return original_rgb, gray_rgb, colorized_rgb, metrics

In [4]:
def test_all_images(input_folder, net, category="test"):
    """
    Test all images in a folder and collect metrics.
    """
    # Get all image files
    image_extensions = ['.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG']
    image_files = []
    
    for file in os.listdir(input_folder):
        if any(file.endswith(ext) for ext in image_extensions):
            image_files.append(file)
    
    if not image_files:
        print(f"No images found in {input_folder}")
        return None
    
    print(f"Found {len(image_files)} images to process.")
    
    results = []
    
    for i, image_file in enumerate(image_files, 1):
        print(f"Processing {i}/{len(image_files)}: {image_file}")
        
        input_path = os.path.join(input_folder, image_file)
        
        try:
            # Process image
            original, grayscale, colorized, metrics = colorize_image_with_metrics(
                input_path, net, compute_metrics=True, original_path=input_path
            )
            
            # Save the colorized output
            output_filename = f"{os.path.splitext(image_file)[0]}_colorized.jpg"
            output_path = os.path.join("Colorization_Dataset/baseline_outputs", output_filename)
            colorized_bgr = cv2.cvtColor(colorized, cv2.COLOR_RGB2BGR)
            cv2.imwrite(output_path, colorized_bgr)
            
            # Store results
            result = {
                'Image_ID': image_file,
                'Category': category,
                'Width': original.shape[1],
                'Height': original.shape[0],
                'MAE': metrics.get('MAE', np.nan),
                'MSE': metrics.get('MSE', np.nan),
                'PSNR': metrics.get('PSNR', np.nan),
                'SSIM': metrics.get('SSIM', np.nan),
                'Delta_E': metrics.get('Delta E', np.nan),
                'Inference_Time_ms': metrics.get('Inference_Time_ms', np.nan)
            }
            results.append(result)
            
            # Save sample visualization for first few images
            if i <= 3:
                plt.figure(figsize=(18, 6))
                plt.subplot(1, 3, 1)
                plt.imshow(original)
                plt.title(f"Original\n{image_file}", fontsize=12)
                plt.axis('off')
                
                plt.subplot(1, 3, 2)
                plt.imshow(grayscale, cmap='gray')
                plt.title("Grayscale Input", fontsize=12)
                plt.axis('off')
                
                plt.subplot(1, 3, 3)
                plt.imshow(colorized)
                plt.title(f"Colorized Output\nPSNR: {metrics.get('PSNR', 0):.2f}dB", fontsize=12)
                plt.axis('off')
                
                plt.tight_layout()
                plt.savefig(f"Result_images/sample_{i}_{os.path.splitext(image_file)[0]}_comparison.png", 
                           dpi=150, bbox_inches='tight')
                plt.close()
                
        except Exception as e:
            print(f"Error processing {image_file}: {e}")
            continue
    
    return pd.DataFrame(results)

In [ ]:
# Test all images in Input_images folder
input_folder = "Input_images"
results_df = test_all_images(input_folder, net, category="test")

# Display summary
if results_df is not None:
    print("\n" + "="*60)
    print("EVALUATION SUMMARY")
    print("="*60)
    print(f"Total images processed: {len(results_df)}")
    print("\nAverage Metrics:")
    print(f"MAE  : {results_df['MAE'].mean():.4f} ± {results_df['MAE'].std():.4f}")
    print(f"MSE  : {results_df['MSE'].mean():.4f} ± {results_df['MSE'].std():.4f}")
    print(f"PSNR : {results_df['PSNR'].mean():.2f} ± {results_df['PSNR'].std():.2f} dB")
    print(f"SSIM : {results_df['SSIM'].mean():.4f} ± {results_df['SSIM'].std():.4f}")
    print(f"Delta E: {results_df['Delta_E'].mean():.2f} ± {results_df['Delta_E'].std():.2f}")
    print(f"Inference Time: {results_df['Inference_Time_ms'].mean():.2f} ± {results_df['Inference_Time_ms'].std():.2f} ms")
    
    # Save results to CSV
    results_df.to_csv("Colorization_Dataset/evaluation_results/colorization_metrics.csv", index=False)
    results_df.to_csv("evaluation_results/colorization_metrics.csv", index=False)
    print("\nMetrics saved to evaluation_results/colorization_metrics.csv")

Found 34 images to process.
Processing 1/34: ready-rey-zTNBumJSOto-unsplash.jpg
Processing 2/34: charan-ttAxbqKiDjI-unsplash.jpg
Processing 3/34: tim-hufner-dqB4c6AX5qE-unsplash.jpg
Processing 4/34: img1.jpg
Processing 5/34: richard-jaimes-s97-KYat9sA-unsplash.jpg
Processing 6/34: yoann-siloine-_T4w3JDm6ug-unsplash.jpg


In [ ]:
def create_evaluation_visualizations(df):
    """
    Create comprehensive visualization plots for the evaluation results.
    """
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    # 1. PSNR Distribution
    ax1 = axes[0, 0]
    ax1.hist(df['PSNR'], bins=10, color='skyblue', edgecolor='black', alpha=0.7)
    ax1.axvline(df['PSNR'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {df["PSNR"].mean():.2f} dB')
    ax1.set_xlabel('PSNR (dB)')
    ax1.set_ylabel('Frequency')
    ax1.set_title('PSNR Distribution')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 2. SSIM Distribution
    ax2 = axes[0, 1]
    ax2.hist(df['SSIM'], bins=10, color='lightgreen', edgecolor='black', alpha=0.7)
    ax2.axvline(df['SSIM'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {df["SSIM"].mean():.4f}')
    ax2.set_xlabel('SSIM')
    ax2.set_ylabel('Frequency')
    ax2.set_title('SSIM Distribution')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # 3. Delta E Distribution
    ax3 = axes[0, 2]
    ax3.hist(df['Delta_E'], bins=10, color='lightcoral', edgecolor='black', alpha=0.7)
    ax3.axvline(df['Delta_E'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {df["Delta_E"].mean():.2f}')
    ax3.set_xlabel('Delta E')
    ax3.set_ylabel('Frequency')
    ax3.set_title('Delta E Distribution (Perceptual Color Difference)')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # 4. Inference Time Distribution
    ax4 = axes[1, 0]
    ax4.hist(df['Inference_Time_ms'], bins=10, color='gold', edgecolor='black', alpha=0.7)
    ax4.axvline(df['Inference_Time_ms'].mean(), color='red', linestyle='--', linewidth=2, 
                label=f'Mean: {df["Inference_Time_ms"].mean():.2f} ms')
    ax4.set_xlabel('Inference Time (ms)')
    ax4.set_ylabel('Frequency')
    ax4.set_title('Inference Time Distribution')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    
    # 5. MAE vs MSE scatter
    ax5 = axes[1, 1]
    scatter = ax5.scatter(df['MAE'], df['MSE'], c=df['PSNR'], cmap='viridis', s=100, alpha=0.7)
    ax5.set_xlabel('MAE')
    ax5.set_ylabel('MSE')
    ax5.set_title('MAE vs MSE (colored by PSNR)')
    plt.colorbar(scatter, ax=ax5, label='PSNR (dB)')
    ax5.grid(True, alpha=0.3)
    
    # 6. Overall Performance Summary Bar Chart
    ax6 = axes[1, 2]
    
    # Normalize metrics for visualization
    metrics = ['PSNR', 'SSIM', 'MAE_inv', 'Delta_E_inv']
    values = [
        df['PSNR'].mean() / 50,  # Normalized PSNR
        df['SSIM'].mean(),
        1 - df['MAE'].mean(),  # Inverted MAE
        1 - min(df['Delta_E'].mean() / 100, 1)  # Inverted Delta E
    ]
    labels = ['PSNR\n(Rel.)', 'SSIM', '1-MAE', '1-DeltaE']
    colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D']
    
    bars = ax6.bar(labels, values, color=colors, edgecolor='black', alpha=0.8)
    ax6.set_ylim(0, 1)
    ax6.set_ylabel('Normalized Score')
    ax6.set_title('Model Performance Summary\n(Higher is Better)')
    
    # Add value labels on bars
    for bar, val in zip(bars, values):
        ax6.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
                f'{val:.3f}', ha='center', va='bottom', fontsize=10)
    
    ax6.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig("evaluation_results/comprehensive_evaluation_plots.png", dpi=300, bbox_inches='tight')
    plt.savefig("Colorization_Dataset/evaluation_results/comprehensive_evaluation_plots.png", dpi=300, bbox_inches='tight')
    plt.show()
    
    print("Comprehensive evaluation plots saved!")

# Create visualizations
if results_df is not None:
    create_evaluation_visualizations(results_df)

In [ ]:
def create_metrics_table(df):
    """
    Create a formatted metrics table for documentation.
    """
    # Sort by PSNR descending
    sorted_df = df.sort_values('PSNR', ascending=False)
    
    # Format for display
    display_df = sorted_df[['Image_ID', 'PSNR', 'SSIM', 'MAE', 'MSE', 'Delta_E', 'Inference_Time_ms']].copy()
    display_df['PSNR'] = display_df['PSNR'].round(2)
    display_df['SSIM'] = display_df['SSIM'].round(4)
    display_df['MAE'] = display_df['MAE'].round(4)
    display_df['MSE'] = display_df['MSE'].round(6)
    display_df['Delta_E'] = display_df['Delta_E'].round(2)
    display_df['Inference_Time_ms'] = display_df['Inference_Time_ms'].round(2)
    
    return display_df

if results_df is not None:
    metrics_table = create_metrics_table(results_df)
    
    # Save table to CSV
    metrics_table.to_csv("evaluation_results/metrics_table.csv", index=False)
    
    # Display table
    print("\n" + "="*80)
    print("DETAILED METRICS PER IMAGE (Sorted by PSNR)")
    print("="*80)
    print(metrics_table.to_string(index=False))
    
    # Create a styled table image
    fig, ax = plt.subplots(figsize=(12, len(metrics_table) * 0.5 + 1))
    ax.axis('tight')
    ax.axis('off')
    
    # Create table
    table_data = metrics_table.head(10).values.tolist()
    columns = metrics_table.columns.tolist()
    
    table = ax.table(cellText=table_data, colLabels=columns, 
                     cellLoc='center', loc='center',
                     colColours=['#4472C4']*len(columns))
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.2, 1.5)
    
    # Color cells based on values
    for i, row in enumerate(table_data):
        for j, val in enumerate(row):
            if j == 2:  # SSIM column
                if val > 0.8:
                    table[(i+1, j)].set_facecolor('#C6EFCE')
                elif val > 0.6:
                    table[(i+1, j)].set_facecolor('#FFEB9C')
                else:
                    table[(i+1, j)].set_facecolor('#FFC7CE')
    
    plt.title('Top 10 Images by PSNR Score', fontsize=14, pad=20)
    plt.tight_layout()
    plt.savefig("evaluation_results/metrics_table.png", dpi=200, bbox_inches='tight')
    plt.close()
    
    print("\nMetrics table saved to evaluation_results/metrics_table.png")

In [ ]:
def generate_summary_statistics(df):
    """
    Generate comprehensive summary statistics for documentation.
    """
    summary = {
        'Metric': [],
        'Mean': [],
        'Std Dev': [],
        'Min': [],
        'Max': [],
        'Median': []
    }
    
    metrics_to_summarize = ['PSNR', 'SSIM', 'MAE', 'MSE', 'Delta_E', 'Inference_Time_ms']
    
    for metric in metrics_to_summarize:
        if metric in df.columns:
            summary['Metric'].append(metric)
            summary['Mean'].append(df[metric].mean())
            summary['Std Dev'].append(df[metric].std())
            summary['Min'].append(df[metric].min())
            summary['Max'].append(df[metric].max())
            summary['Median'].append(df[metric].median())
    
    summary_df = pd.DataFrame(summary)
    
    # Format for display
    format_dict = {
        'PSNR': '{:.2f}',
        'SSIM': '{:.4f}',
        'MAE': '{:.4f}',
        'MSE': '{:.6f}',
        'Delta_E': '{:.2f}',
        'Inference_Time_ms': '{:.2f}'
    }
    
    for metric, fmt in format_dict.items():
        if metric in summary_df['Metric'].values:
            idx = summary_df[summary_df['Metric'] == metric].index[0]
            for col in ['Mean', 'Std Dev', 'Min', 'Max', 'Median']:
                summary_df.loc[idx, col] = fmt.format(summary_df.loc[idx, col])
    
    return summary_df

if results_df is not None:
    summary_stats = generate_summary_statistics(results_df)
    summary_stats.to_csv("evaluation_results/summary_statistics.csv", index=False)
    
    print("\n" + "="*60)
    print("SUMMARY STATISTICS")
    print("="*60)
    print(summary_stats.to_string(index=False))

In [ ]:
def create_performance_metrics_chart(df):
    """
    Create a bar chart showing all metrics with error bars.
    """
    metrics = ['PSNR', 'SSIM', 'MAE_inv', 'Delta_E_inv']
    labels = ['PSNR (dB)', 'SSIM', '1-MAE', '1-DeltaE']
    
    # Normalize metrics for comparison
    psnr_norm = df['PSNR'] / df['PSNR'].max()
    ssim_norm = df['SSIM']
    mae_inv = 1 - df['MAE']
    delta_e_inv = 1 - (df['Delta_E'] / df['Delta_E'].max())
    
    values = [psnr_norm.mean(), ssim_norm.mean(), mae_inv.mean(), delta_e_inv.mean()]
    errors = [psnr_norm.std(), ssim_norm.std(), mae_inv.std(), delta_e_inv.std()]
    
    fig, ax = plt.subplots(figsize=(10, 6))
    colors = ['#4CAF50', '#2196F3', '#FF9800', '#9C27B0']
    
    bars = ax.bar(labels, values, yerr=errors, capsize=5, color=colors, 
                  edgecolor='black', alpha=0.8, error_kw={'ecolor': 'red', 'linewidth': 2})
    
    ax.set_ylim(0, 1.1)
    ax.set_ylabel('Normalized Score')
    ax.set_title('Normalized Performance Metrics\n(Higher is Better)', fontsize=14)
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add value labels
    for bar, val, err in zip(bars, values, errors):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
                f'{val:.3f}\n±{err:.3f}', ha='center', va='bottom', fontsize=10)
    
    plt.tight_layout()
    plt.savefig("evaluation_results/performance_metrics_bar_chart.png", dpi=300, bbox_inches='tight')
    plt.show()
    
    print("Performance metrics chart saved!")

if results_df is not None:
    create_performance_metrics_chart(results_df)

In [ ]:
def create_box_plots(df):
    """
    Create box plots for all metrics to show distribution.
    """
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    metrics_to_plot = ['PSNR', 'SSIM', 'MAE', 'MSE', 'Delta_E', 'Inference_Time_ms']
    titles = ['PSNR (dB)', 'SSIM', 'MAE', 'MSE', 'Delta E', 'Inference Time (ms)']
    
    for i, (metric, title) in enumerate(zip(metrics_to_plot, titles)):
        row, col = i // 3, i % 3
        ax = axes[row, col]
        
        # Box plot with outliers
        bp = ax.boxplot(df[metric].dropna(), patch_artist=True)
        bp['boxes'][0].set_facecolor('#4472C4')
        bp['boxes'][0].set_alpha(0.7)
        
        ax.set_title(title, fontsize=12)
        ax.set_ylabel(title)
        ax.grid(True, alpha=0.3)
        
        # Add mean value annotation
        mean_val = df[metric].mean()
        ax.axhline(mean_val, color='red', linestyle='--', linewidth=1.5, 
                   label=f'Mean: {mean_val:.3f}')
        ax.legend()
    
    plt.tight_layout()
    plt.savefig("evaluation_results/metrics_box_plots.png", dpi=300, bbox_inches='tight')
    plt.show()
    
    print("Box plots saved!")

if results_df is not None:
    create_box_plots(results_df)

In [ ]:
def create_correlation_heatmap(df):
    """
    Create a correlation heatmap for all metrics.
    """
    metrics_corr = ['PSNR', 'SSIM', 'MAE', 'MSE', 'Delta_E', 'Inference_Time_ms']
    corr_df = df[metrics_corr].corr()
    
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Create heatmap
    im = ax.imshow(corr_df.values, cmap='RdBu_r', vmin=-1, vmax=1)
    
    # Add labels
    ax.set_xticks(np.arange(len(metrics_corr)))
    ax.set_yticks(np.arange(len(metrics_corr)))
    ax.set_xticklabels(metrics_corr, rotation=45, ha='right')
    ax.set_yticklabels(metrics_corr)
    
    # Add correlation values
    for i in range(len(metrics_corr)):
        for j in range(len(metrics_corr)):
            text = ax.text(j, i, f'{corr_df.iloc[i, j]:.3f}',
                          ha='center', va='center', color='black' if abs(corr_df.iloc[i, j]) < 0.5 else 'white',
                          fontsize=10)
    
    plt.title('Correlation Matrix of Evaluation Metrics', fontsize=14)
    plt.colorbar(im, ax=ax, label='Correlation Coefficient')
    plt.tight_layout()
    plt.savefig("evaluation_results/metrics_correlation_heatmap.png", dpi=300, bbox_inches='tight')
    plt.show()
    
    print("Correlation heatmap saved!")

if results_df is not None:
    create_correlation_heatmap(results_df)

In [ ]:
def generate_final_report(df):
    """
    Generate a comprehensive final report with all metrics and visualizations.
    """
    from datetime import datetime
    
    report = []
    report.append("="*70)
    report.append("COLORIZATION MODEL EVALUATION REPORT")
    report.append("="*70)
    report.append(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    report.append(f"Total Images Evaluated: {len(df)}")
    report.append("")
    
    report.append("-"*70)
    report.append("SUMMARY STATISTICS")
    report.append("-"*70)
    
    for metric in ['PSNR', 'SSIM', 'MAE', 'MSE', 'Delta_E', 'Inference_Time_ms']:
        if metric in df.columns:
            report.append(f"{metric:20s} | "
                         f"Mean: {df[metric].mean():.4f} | "
                         f"Std: {df[metric].std():.4f} | "
                         f"Min: {df[metric].min():.4f} | "
                         f"Max: {df[metric].max():.4f}")
    
    report.append("")
    report.append("-"*70)
    report.append("PERFORMANCE RANGES")
    report.append("-"*70)
    report.append(f"PSNR Range: {df['PSNR'].min():.2f} - {df['PSNR'].max():.2f} dB")
    report.append(f"SSIM Range: {df['SSIM'].min():.4f} - {df['SSIM'].max():.4f}")
    report.append(f"Delta E Range: {df['Delta_E'].min():.2f} - {df['Delta_E'].max():.2f}")
    report.append(f"Inference Time Range: {df['Inference_Time_ms'].min():.2f} - {df['Inference_Time_ms'].max():.2f} ms")
    
    report.append("")
    report.append("-"*70)
    report.append("BEST AND WORST PERFORMING IMAGES")
    report.append("-"*70)
    
    # Best PSNR
    best_idx = df['PSNR'].idxmax()
    best_row = df.loc[best_idx]
    report.append(f"Best PSNR: {best_row['Image_ID']} ({best_row['PSNR']:.2f} dB)")
    
    # Worst PSNR
    worst_idx = df['PSNR'].idxmin()
    worst_row = df.loc[worst_idx]
    report.append(f"Worst PSNR: {worst_row['Image_ID']} ({worst_row['PSNR']:.2f} dB)")
    
    # Best Delta E (lowest)
    best_de_idx = df['Delta_E'].idxmin()
    best_de_row = df.loc[best_de_idx]
    report.append(f"Best Delta E: {best_de_row['Image_ID']} ({best_de_row['Delta_E']:.2f})")
    
    # Worst Delta E (highest)
    worst_de_idx = df['Delta_E'].idxmax()
    worst_de_row = df.loc[worst_de_idx]
    report.append(f"Worst Delta E: {worst_de_row['Image_ID']} ({worst_de_row['Delta_E']:.2f})")
    
    report.append("")
    report.append("-"*70)
    report.append("END OF REPORT")
    report.append("-"*70)
    
    # Save report
    report_text = "\n".join(report)
    with open("evaluation_results/evaluation_report.txt", "w") as f:
        f.write(report_text)
    
    print(report_text)
    print("\nReport saved to evaluation_results/evaluation_report.txt")

if results_df is not None:
    generate_final_report(results_df)

In [ ]:
# List all generated files
print("\n" + "="*60)
print("GENERATED FILES")
print("="*60)

files = []
for root, dirs, filenames in os.walk("evaluation_results"):
    for filename in filenames:
        filepath = os.path.join(root, filename)
        files.append(filepath)

for file in sorted(files):
    size = os.path.getsize(file) / 1024  # KB
    print(f"📁 {file} ({size:.2f} KB)")

In [ ]:
print("\n" + "="*60)
print("EXECUTION COMPLETE!")
print("="*60)
print(f"✅ Processed {len(results_df)} images")
print(f"✅ Generated evaluation metrics")
print(f"✅ Created visualization plots")
print(f"✅ Saved reports and tables")
print("\n📂 Output Directory: evaluation_results/")
print("\n🎯 Next Steps for Documentation:")
print("   1. Use the generated plots in your research documentation")
print("   2. Include the metrics tables in your evaluation section")
print("   3. Reference the summary statistics in your results discussion")
print("   4. Use the comparison visualizations to demonstrate model performance")